<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 40%, #0d2137 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(88,166,255,0.1); border: 1px solid rgba(88,166,255,0.3); color: #58a6ff; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 1</span>
  <h1 style="color: #e6edf3; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Kafka Cluster Setup & Validation</h1>
  <p style="color: #8b949e; font-size: 1.1em;">Spin up a full Confluent Kafka ecosystem with Docker and validate every component is healthy.</p>
</div>

---

## 🎯 Overview

In this lab, you will spin up a complete **Confluent Kafka ecosystem** using Docker Compose and validate that every component is working correctly. 

By the end of this lab, you will have hands-on experience with:
- 🐳 Starting a multi-component Kafka cluster
- 🔍 Verifying Zookeeper, Kafka Broker, Schema Registry, and Kafka Connect
- 📋 Creating Kafka topics and inspecting their configuration
- 🚀 Producing and consuming messages end-to-end

---

## ⚙️ Prerequisites

Make sure the following are installed on your machine before starting:

| Tool | Minimum Version | Check Command |
|---|---|---|
| **Docker Desktop** | 4.x+ | `docker --version` |
| **Docker Compose** | v2+ | `docker compose version` |

<div style="background-color: rgba(88, 166, 255, 0.1); border-left: 4px solid #58a6ff; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>📌 Note:</strong> Ensure Docker Desktop is running before proceeding.
</div>

---

## 🏗️ Architecture

The Docker Compose stack spins up the following services:

<div style="text-align:center; margin: 20px 0;">
  <img src="kafka_architecture.png" alt="Kafka Ecosystem Architecture" style="max-width:100%; border-radius: 12px; border: 1px solid #30363d; background: #fff; padding: 8px;"/>
</div>

---

## <span style="color: #58a6ff;">Step 1:</span> Start the Kafka Cluster

Execute the cell below to start all services in detached mode (`-d`). 

<div style="background-color: rgba(63, 185, 80, 0.1); border-left: 4px solid #3fb950; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>💡 Tip:</strong> The `-d` flag runs containers in the background. Without it, logs would lock up the cell execution.
</div>

In [11]:
!docker-compose up -d

time="2026-05-17T19:16:33+05:30" level=warning msg="d:\\trainings\\CCDAK-2\\kafka_training\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container kafka_training-zookeeper-1  Starting
 Container kafka_training-zookeeper-1  Started
 Container kafka_training-kafka-1  Starting
 Container kafka_training-kafka-1  Started
 Container kafka_training-schema-registry-1  Starting
 Container kafka_training-schema-registry-1  Started
 Container kafka_training-kafka-connect-1  Starting
 Container kafka_training-kafka-connect-1  Started
 Container kafka_training-control-center-1  Starting
 Container kafka_training-control-center-1  Started


---

## <span style="color: #58a6ff;">Step 2:</span> Verify All Containers Are Running

Check the status of all running containers. 

**All 5 containers should show an "Up" status.** If any show `Exited`, check logs using `docker logs <container-name>`.

In [12]:
!docker ps

CONTAINER ID   IMAGE                                             COMMAND                  CREATED         STATUS                            PORTS                                         NAMES
a69cfd40789a   confluentinc/cp-enterprise-control-center:7.5.0   "/etc/confluent/dock…"   4 minutes ago   Up 7 seconds                      0.0.0.0:9021->9021/tcp, [::]:9021->9021/tcp   kafka_training-control-center-1
86ffa6a922f2   confluentinc/cp-kafka-connect:7.5.0               "/etc/confluent/dock…"   4 minutes ago   Up 7 seconds (health: starting)   0.0.0.0:8083->8083/tcp, [::]:8083->8083/tcp   kafka_training-kafka-connect-1
8b13d1a7a3c8   confluentinc/cp-schema-registry:7.5.0             "/etc/confluent/dock…"   4 minutes ago   Up 7 seconds                      0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp   kafka_training-schema-registry-1
80a15e764cdc   confluentinc/cp-kafka:7.5.0                       "/etc/confluent/dock…"   4 minutes ago   Up 8 seconds                      0.0.0.0:9092->

---

## <span style="color: #58a6ff;">Step 3:</span> Validate Zookeeper

Zookeeper is the coordination service for Kafka. Let's verify it is running and Kafka has registered itself.

<div style="background-color: rgba(63, 185, 80, 0.1); border: 1px solid rgba(63, 185, 80, 0.3); padding: 10px 15px; margin: 15px 0; border-radius: 8px; color: #3fb950;">
  <strong>✅ Success Criteria:</strong> Look for <code>brokers</code> and <code>controller</code> in the output list. This confirms Kafka is registered with Zookeeper.
</div>

In [14]:
!docker exec kafka_training-zookeeper-1 bash -c "zookeeper-shell localhost:2181 ls /"

Connecting to localhost:2181

WATCHER::

WatchedEvent state:SyncConnected type:None path:null
[admin, brokers, cluster, config, consumers, controller, controller_epoch, feature, isr_change_notification, latest_producer_id_block, log_dir_event_notification, zookeeper]


---

## <span style="color: #58a6ff;">Step 4:</span> Validate the Kafka Broker

Verify the Kafka broker is accepting connections and responding to API requests.

<div style="background-color: rgba(63, 185, 80, 0.1); border: 1px solid rgba(63, 185, 80, 0.3); padding: 10px 15px; margin: 15px 0; border-radius: 8px; color: #3fb950;">
  <strong>✅ Success Criteria:</strong> You should see <code>localhost:9092 (id: 1 rack: null)</code> followed by a list of supported APIs in the output.
</div>

In [15]:
!docker exec kafka_training-kafka-1 kafka-broker-api-versions --bootstrap-server localhost:9092

localhost:9092 (id: 1 rack: null) -> (
	Produce(0): 0 to 9 [usable: 9],
	Fetch(1): 0 to 15 [usable: 15],
	ListOffsets(2): 0 to 8 [usable: 8],
	Metadata(3): 0 to 12 [usable: 12],
	LeaderAndIsr(4): 0 to 7 [usable: 7],
	StopReplica(5): 0 to 4 [usable: 4],
	UpdateMetadata(6): 0 to 8 [usable: 8],
	ControlledShutdown(7): 0 to 3 [usable: 3],
	OffsetCommit(8): 0 to 8 [usable: 8],
	OffsetFetch(9): 0 to 8 [usable: 8],
	FindCoordinator(10): 0 to 4 [usable: 4],
	JoinGroup(11): 0 to 9 [usable: 9],
	Heartbeat(12): 0 to 4 [usable: 4],
	LeaveGroup(13): 0 to 5 [usable: 5],
	SyncGroup(14): 0 to 5 [usable: 5],
	DescribeGroups(15): 0 to 5 [usable: 5],
	ListGroups(16): 0 to 4 [usable: 4],
	SaslHandshake(17): 0 to 1 [usable: 1],
	ApiVersions(18): 0 to 3 [usable: 3],
	CreateTopics(19): 0 to 7 [usable: 7],
	DeleteTopics(20): 0 to 6 [usable: 6],
	DeleteRecords(21): 0 to 2 [usable: 2],
	InitProducerId(22): 0 to 4 [usable: 4],
	OffsetForLeaderEpoch(23): 0 to 4 [usable: 4],
	AddPartitionsToTxn(24): 0 to 3 [usable

---

## <span style="color: #58a6ff;">Step 8:</span> Validate Schema Registry

The Schema Registry stores and manages Avro/JSON/Protobuf schemas. Test it is accessible.

In [17]:
!docker exec kafka_training-schema-registry-1 curl -s http://localhost:8081/subjects

[]


---

## <span style="color: #58a6ff;">Step 9:</span> Validate Kafka Connect

Kafka Connect is a framework for streaming data between Kafka and external systems. Verify it is running.

In [18]:
!docker exec kafka_training-kafka-connect-1 curl -s http://localhost:8083/connectors

[]


---

## <span style="color: #58a6ff;">Step 10:</span> Access Confluent Control Center (UI)

<div style="background-color: rgba(63, 185, 80, 0.1); border: 1px solid rgba(63, 185, 80, 0.3); padding: 10px 15px; margin: 15px 0; border-radius: 8px; color: #3fb950;">
  <strong>✅ You will only be able to access Control Center when you will run local setup
</div>


Confluent Control Center provides a web UI for monitoring and managing your Kafka cluster.

1. Open your browser and navigate to: <a href="http://localhost:9021" target="_blank"><strong>http://localhost:9021</strong></a>
2. You should see the **Control Center** dashboard.

**Things to explore in the UI:**
- **Brokers** — View broker health and metrics
- **Topics** — See `test-topic` and its partition details
- **Schema Registry** — View registered schemas
- **Connect** — Manage Kafka connectors

---

<div style="background-color: rgba(88, 166, 255, 0.1); border: 1px solid rgba(88, 166, 255, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #58a6ff; margin-bottom: 10px;">🎉 Lab 1 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You have successfully deployed and validated a complete Kafka cluster.</p>
</div>